# Session 9 — Capstone Part 1: Real Networks, Real Groups

**Goal of this session:** apply the null-model and community-detection machinery from sessions 1 through 3 to real people, using data returning viewers will already recognise.

*Network Neuroscience in Python, session 9 of 10.*

## Why this matters

Everything from session 1 onward has been tested first on a network with a known ground truth, exactly so we could trust it before pointing it at anything real. This session is where that trust gets spent: the same null-model test from session 1, and the same community detection from session 3, run on genuine human brain networks instead of our toy one. Session 10 takes it further, with a properly controlled group comparison and a hypothesis stated in advance.

## The data: fifty people, one more time

This is the exact dataset the first course's capstone used — 50 participants (25 children, 25 adults) watching a short film in the scanner, reduced to 39 region time series with the MSDL atlas. It's cached in `data/dev_fmri_timeseries.npz` in *this* repo too, copied straight from [python-for-neuroscience](https://github.com/saeedrafsharx/python-for-neuroscience)'s own cache rather than re-extracted, so returning viewers are looking at literally the same numbers as course 1's finale.

In [ ]:
import io
import os
import urllib.request

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats

REPO_RAW = "https://raw.githubusercontent.com/saeedrafsharx/network-neuroscience-python/main/data/"
DATA = "../data/" if os.path.exists("../data/dev_fmri_timeseries.npz") else REPO_RAW
print("reading data from:", DATA)


def load_npz(name):
    if DATA.startswith("http"):
        with urllib.request.urlopen(DATA + name) as response:
            return np.load(io.BytesIO(response.read()), allow_pickle=True)
    return np.load(DATA + name, allow_pickle=True)


d = load_npz("dev_fmri_timeseries.npz")
series = d["timeseries"]
group = d["group"]
age = d["age"]
labels = [str(x) for x in d["regions"]]
networks = [str(x) for x in d["networks"]]

print("time series:", series.shape, "= participants x timepoints x regions")
print("children:", (group == "child").sum(), " adults:", (group == "adult").sum())

## One connectivity matrix, one graph, per person

Same as session 12 of the first course: correlate every pair of regions per participant, then threshold at r > 0.45 to get a binary graph. That threshold is a specific, disclosed choice — session 6 should have made you suspicious of any number here that isn't.

In [ ]:
conn = np.array([np.corrcoef(subject.T) for subject in series])
print("connectivity matrices:", conn.shape, "= participants x regions x regions")

THRESHOLD = 0.45


def build_graph(matrix, threshold=THRESHOLD, node_labels=labels):
    adjacency = (matrix > threshold) & ~np.eye(matrix.shape[0], dtype=bool)
    g = nx.from_numpy_array(adjacency.astype(int))
    return nx.relabel_nodes(g, dict(enumerate(node_labels)))


participant_graphs = [build_graph(m) for m in conn]
edge_counts = [g.number_of_edges() for g in participant_graphs]
print(f"edges per participant: mean={np.mean(edge_counts):.1f}, "
      f"range [{min(edge_counts)}, {max(edge_counts)}]")

## The null-model check, on a real brain

Session 1's whole point was: a graph metric means nothing without something to compare it to. Let's find out whether one real participant's clustering coefficient is higher than a degree-preserving null predicts — the exact test from session 1, unchanged, pointed at a real network instead of our toy one. We use clustering coefficient specifically because, unlike path length, it stays well defined even if a participant's thresholded graph isn't fully connected (individual, noisier single-subject graphs often aren't, unlike the smoother group averages from course 1's session 12).

In [ ]:
def degree_preserving_null(G, n_swaps_per_edge=10, seed=0):
    rng = np.random.default_rng(seed)
    G_null = G.copy()
    n_swaps = max(10, n_swaps_per_edge * G.number_of_edges())
    nx.double_edge_swap(G_null, nswap=n_swaps, max_tries=n_swaps * 20,
                         seed=int(rng.integers(1_000_000_000)))
    return G_null


def test_against_null(G, metric_fn, n_random=500, seed=0):
    rng = np.random.default_rng(seed)
    observed = metric_fn(G)
    null_vals = np.array([metric_fn(degree_preserving_null(G, seed=int(rng.integers(1_000_000_000))))
                           for _ in range(n_random)])
    null_mean, null_std = null_vals.mean(), null_vals.std()
    z = (observed - null_mean) / null_std if null_std > 0 else np.nan
    p = (np.sum(np.abs(null_vals - null_mean) >= np.abs(observed - null_mean)) + 1) / (n_random + 1)
    return observed, null_vals, z, p


participant_idx = int(np.where(group == "adult")[0][0])  # first adult in the file, chosen before looking at the result
one_graph = participant_graphs[participant_idx]
print(f"participant {participant_idx}: group={group[participant_idx]}, age={age[participant_idx]:.1f}, "
      f"{one_graph.number_of_edges()} edges, connected={nx.is_connected(one_graph)}")

observed, null_vals, z, p = test_against_null(one_graph, nx.average_clustering, n_random=500, seed=1)
print(f"\nclustering coefficient: observed={observed:.3f}, null mean={null_vals.mean():.3f}, "
      f"z={z:+.2f}, p={p:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.hist(null_vals, bins=30, color="#a0aec0", edgecolor="white", label="null distribution")
ax.axvline(observed, color="#c53030", linewidth=3, label=f"observed = {observed:.3f}")
ax.set_xlabel("clustering coefficient", fontsize=12)
ax.set_ylabel("count", fontsize=12)
ax.set_title(f"Participant {participant_idx} ({group[participant_idx]}) vs. its own degree-preserving null",
             fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

This one participant's clustering is well above what their own degree sequence predicts — the same pattern session 1 found in the toy network, now confirmed on a real functional brain network. That's reassuring: it means the machinery built on synthetic data is picking up something real, not an artefact of how we constructed the toy example.

## Community detection on the group averages

Now session 3's Louvain algorithm, on the children's and adults' group-average connectivity matrices. Group averages are smoother than any individual matrix, so we can afford a lower, more inclusive threshold (0.3, chosen and stated here — not 0.45) and still get a connected-enough graph for community detection to say something meaningful, rather than fragmenting into mostly-singleton "communities".

In [ ]:
COMMUNITY_THRESHOLD = 0.3

mean_child = conn[group == "child"].mean(axis=0)
mean_adult = conn[group == "adult"].mean(axis=0)

g_child = build_graph(mean_child, threshold=COMMUNITY_THRESHOLD)
g_adult = build_graph(mean_adult, threshold=COMMUNITY_THRESHOLD)

comm_child = nx.community.louvain_communities(g_child, seed=0)
comm_adult = nx.community.louvain_communities(g_adult, seed=0)

print(f"children: {g_child.number_of_edges()} edges, {len(comm_child)} communities, "
      f"Q={nx.community.modularity(g_child, comm_child):.3f}")
print(f"adults:   {g_adult.number_of_edges()} edges, {len(comm_adult)} communities, "
      f"Q={nx.community.modularity(g_adult, comm_adult):.3f}")

## Do the modules look different?

Same layout for both (positions from the averaged child+adult graph, so a region sits in the same spot in both panels), coloured by detected community. Watch where the four **DMN** regions (`L DMN`, `R DMN`, `Front DMN`, `Med DMN`) land in each.

In [ ]:
palette = ["#2b6cb0", "#dd6b20", "#805ad5", "#38a169", "#e53e3e", "#d69e2e", "#319795", "#b83280"]
pos = nx.spring_layout(build_graph((mean_child + mean_adult) / 2, threshold=COMMUNITY_THRESHOLD), seed=2)
dmn_regions = {"L DMN", "R DMN", "Front DMN", "Med DMN"}

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
for ax, g, comms, title in zip(axes, [g_child, g_adult], [comm_child, comm_adult],
                                ["Children", "Adults"]):
    node_to_colour = {n: palette[i % len(palette)] for i, c in enumerate(comms) for n in c}
    colours = [node_to_colour[n] for n in g.nodes()]
    sizes = [900 if n in dmn_regions else 420 for n in g.nodes()]
    nx.draw_networkx_edges(g, pos, alpha=0.3, ax=ax)
    nx.draw_networkx_nodes(g, pos, node_color=colours, node_size=sizes,
                            edgecolors=["black" if n in dmn_regions else "white" for n in g.nodes()],
                            linewidths=[2.5 if n in dmn_regions else 0.8 for n in g.nodes()], ax=ax)
    dmn_labels = {n: n for n in g.nodes() if n in dmn_regions}
    nx.draw_networkx_labels(g, pos, labels=dmn_labels, font_size=10, font_weight="bold", ax=ax)
    ax.set_title(f"{title}: {len(comms)} communities, r > {COMMUNITY_THRESHOLD}", fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

for name, comms in [("children", comm_child), ("adults", comm_adult)]:
    dmn_comm_ids = {i for i, c in enumerate(comms) for n in c if n in dmn_regions}
    n = len(dmn_comm_ids)
    print(f"{name}: the 4 DMN regions fall into {n} detected {'community' if n == 1 else 'communities'}")

## What we're looking at

In this sample, the four default mode regions land in a single shared community in the adult group average, but split across separate communities in the children's. That's a real, specific pattern, not just "the pictures look different": the default mode network reads as one coherent module in adults and a less unified one in children — consistent with course 1 session 12's finding that within-DMN connectivity strengthens with age, now showing up as a difference in module *membership* rather than just average correlation strength.

One picture is not a result. It's a single split of 50 participants into two group averages, at one threshold, with no statistical test attached yet. Session 10 is where we do that properly.

**Next session:** the actual hypothesis test — hub cartography and rich-club analysis, compared between groups with the density-controlled method from session 6, stated as a prediction before we run anything.